# Tune quick guide with autolog disabled

## Introduction
In this notebook, you'll see how to perform Hyperparameter tuning tasks with FLAML for different scenarios. We'll have mlflow autolog disabled, and show you how to log metrics manually.

The scenarios are as below:

1. Tune a pyspark ml type model

    In this scenario, we'll tune a SynapseML lightGBM model which is pyspark ml type. For the mlflow integration, we'll not set mlflow experiment name and run name; the experiment name will be the notebook name by default. We'll also log some metrics manually.

2. Tune a non spark model

    In this scenario, we'll tune an original lightGBM model which is sklearn type, i.e., non spark type. And we'll customize mlflow experiment name and run name, and log flaml pre-defined metrics.

    *It's not recommended to log metrics manually when logging flaml pre-defined metrics.*

    *When use_spark=True, manually logging inside the train function will fail as mlflow endpoint is not set in executors, extra configs are needed to make it work.*

Please ref [FLAML doc](https://microsoft.github.io/FLAML/docs/Getting-Started/) for more details of FLAML usage.

## Prerequisites
We need to install flaml for performing automl tasks.

In [ ]:
%pip install "flaml[synapse]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl"

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, -1, Finished, Available)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 1.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 3.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 39.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 27.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 35.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 38.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 25.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 45.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0 -> 23.1.1
[notice] To update, run: /nfs4/pyenv-42210f2b-5d0a-4833-adb8-acc97d34eb5b/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



## Case 1. Tune a pyspark ml type model

In this scenario, we'll tune a SynapseML lightGBM model which is pyspark ml type. For the mlflow integration, we'll not set mlflow experiment name and run name; the experiment name will be the notebook name by default. We'll also log some metrics manually.

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/demo-images/tune_autolog_off_1.png)


In [ ]:
import mlflow
import flaml
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

import pyspark
from pyspark.ml.feature import VectorAssembler
from synapse.ml.lightgbm import LightGBMRegressor
from synapse.ml.train import ComputeModelStatistics

mlflow.autolog(disable=True)  # disable mlflow autologging

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 9, Finished, Available)

### Prepare Dataset

In [ ]:
data = fetch_california_housing()

feature_cols = ["f" + str(i) for i in range(data.data.shape[1])]
header = ["target"] + feature_cols
df = spark.createDataFrame(pd.DataFrame(data=np.column_stack((data.target, data.data)), columns=header)).repartition(1)
print("Dataframe has {} rows".format(df.count()))

# Convert features into a single vector column
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = featurizer.transform(df)["target", "features"]

train_data, test_data = data.randomSplit([0.85, 0.15], seed=41)

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 10, Finished, Available)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:604: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.


### Define training function

In [ ]:
def train(config):
    """
    This train() function:
     - takes hyperparameters config as inputs (for tuning later)
     - returns the R^2 score on the test dataset

    Wrapping code as a function makes it easier to reuse the code later with FLAML.
    """
    lgr = LightGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
    )
    model = lgr.fit(train_data)
    # Define an evaluation metric and evaluate the model on the test dataset.
    predictions = model.transform(test_data)
    cms = ComputeModelStatistics(
        evaluationMetric="regression", labelCol="target", scoresCol="prediction"
    )
    metrics = cms.transform(predictions).collect()[0].asDict()

    # log metrics with mlflow
    with mlflow.start_run(nested=True):
        mlflow.log_metric("MSE", metrics["mean_squared_error"])
        mlflow.log_metric("RMSE", metrics["root_mean_squared_error"])
        mlflow.log_metric("R2", metrics["R^2"])
        mlflow.log_metric("MAE", metrics["mean_absolute_error"])

    return {"r2": metrics["R^2"]}

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 11, Finished, Available)

### Define search space

In [ ]:
params = {
    "alpha": flaml.tune.uniform(0, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 12, Finished, Available)

### Run tuning

In [ ]:
# no need to set use_spark since a spark model itself will run in parallel
analysis = flaml.tune.run(
    train,
    params,
    metric="r2",
    mode="max",
    num_samples=5,
)

mlflow.end_run()  # end current run

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 13, Finished, Available)

[flaml.tune.tune: 04-25 11:24:29] {530} INFO - Using search algorithm BlendSearch.
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
You passed a `space` parameter to OptunaSearch that contained unresolved search space definitions. OptunaSearch should however be instantiated with fully configured search spaces only. To use Ray Tune's automatic search space conversion, pass the space definition as part of the `config` argument to `tune.run()` instead.
[flaml.tune.tune: 04-25 11:24:29] {809} INFO - trial 1 config: {'alpha': 0.09743207287894917, 'learningRate': 0.64761881525086, 'numLeaves': 30, 'numIterations': 172}
[flaml.tune.tune: 04-25 11:24:37] {809} INFO - trial 2 config: {'alpha': 0.771320643266746, 'learningRate': 0.021731197410042098, 'numLeaves': 74, 'numI

In [ ]:
synapselgb_config = analysis.best_config
synapselgb_r2 = analysis.best_result['r2']
print(f"Best config: {synapselgb_config}")
print(f"R^2: {synapselgb_r2}")

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 14, Finished, Available)

Best config: {'alpha': 0.5940316589938806, 'learningRate': 0.22926504794631342, 'numLeaves': 35, 'numIterations': 279}
R^2: 0.8094330941991653



## Case 2. Tune a non spark model

In this scenario, we'll tune an original lightGBM model which is sklearn type, i.e., non spark type. And we'll customize mlflow experiment name and run name, and log some metrics manually.

*It's not recommended to log metrics manually when logging flaml pre-defined metrics;*

*When use_spark=True, manually logging inside the train function will fail as mlflow endpoint is not set in executors, extra configs are needed to make it work.*

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/demo-images/tune_exp_2.png)


### Prepare Dataset

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.15)

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 15, Finished, Available)

### Define training function

In [ ]:
def train_lgb(config):
    lgr = LGBMRegressor(
        objective="quantile",
        alpha=config["alpha"],
        learningRate=config["learningRate"],
        numLeaves=config["numLeaves"],
        labelCol="target",
        numIterations=config["numIterations"],
    )
    model = lgr.fit(train_x, train_y, eval_metric=["l2"], eval_set=[(train_x, train_y)])
    # Define an evaluation metric and evaluate the model on the test dataset.
    pred_y = model.predict(test_x)
    r2 = r2_score(test_y, pred_y)
    mse = mean_squared_error(test_y, pred_y)
    mae = mean_absolute_error(test_y, pred_y)

    # # It's not recommended to log metrics manually and log flaml pre-defined metrics in the same time
    # # Below 4 lines of code is needed when the following manually logging is enabled and use_spark=True 
    # from synapse.ml.mlflow import set_mlflow_env_config
    # set_mlflow_env_config(mlflow_env_config)
    # if mlflow_exp_name is not None:
    #     mlflow.set_experiment(mlflow_exp_name)

    # with mlflow.start_run(nested=True):
    #     mlflow.log_metric("MSE", mse)
    #     mlflow.log_metric("R2", r2)
    #     mlflow.log_metric("MAE", mae)

    return {"r2": r2}

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 19, Finished, Available)

### Run tuning

In [ ]:
mlflow_exp_name = "tune_exp"
mlflow.set_experiment(mlflow_exp_name)  # customize the experiment name
with mlflow.start_run(nested=True, run_name="tune_run"):  # customize the run name
    # # Below 2 lines of code is needed when the manually logging in train_lgb is enabled and use_spark=True 
    # from synapse.ml.mlflow import get_mlflow_env_config
    # mlflow_env_config = get_mlflow_env_config()

    analysis = flaml.tune.run(
        train_lgb,
        params,
        metric="r2",
        mode="max",
        num_samples=5,
        use_spark=True,  # use spark to parallelize the training
        n_concurrent_trials=2,
        mlflow_exp_name=mlflow_exp_name,
    )

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 20, Finished, Available)

[Parallel(n_jobs=2)]: Using backend SparkDistributedBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done   1 out of   1 | elapsed:    1.2s finished
[flaml.tune.tune: 04-25 11:26:55] {530} INFO - Using search algorithm BlendSearch.
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
You passed a `space` parameter to OptunaSearch that contained unresolved search space definitions. OptunaSearch should however be instantiated with fully configured search spaces only. To use Ray Tune's automatic search space conversion, pass the space definition as part of the `config` argument to `tune.run()` instead.
[flaml.tune.tune: 04-25 11:26:55] {719} INFO - Number of trials: 2/5, 2 RUNNING, 0 TERMINATED
[Parall

In [ ]:
lgb_config = analysis.best_config
lgb_r2 = analysis.best_result['r2']
print(f"Best config: {lgb_config}")
print(f"R^2: {lgb_r2}")

StatementMeta(, 5ac1af0f-c427-4d5f-a115-757827ae6d7c, 21, Finished, Available)

Best config: {'alpha': 0.5940316589938806, 'learningRate': 0.22926504794631342, 'numLeaves': 35, 'numIterations': 279}
R^2: 0.8162562768756668
